# 🎭 Sentiment Analysis on Movie Reviews

**Author:** Md Arif Hasan Badsha  
**Dataset:** NLTK Movie Reviews (Pang & Lee, 2002)  
**Goal:** Compare multiple ML models for binary sentiment classification (positive vs. negative)

---

## 📌 Project Overview

This notebook walks through a complete NLP pipeline:
1. Data loading & exploration
2. Text preprocessing
3. Feature extraction (Bag of Words + TF-IDF)
4. Model training & comparison
5. Evaluation with metrics + visualizations
6. Error analysis

---

## 1. Setup & Imports

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
import re
import string
from nltk.corpus import movie_reviews, stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Feature extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Download required NLTK data
for pkg in ['movie_reviews', 'stopwords', 'wordnet', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

print("✅ All imports successful")

## 2. Data Loading & Exploration

In [ ]:
# Load the movie reviews dataset
# Dataset: 2000 movie reviews labelled as 'pos' or 'neg'

documents = []
labels = []

for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        raw_text = movie_reviews.raw(fileid)
        documents.append(raw_text)
        labels.append(category)

df = pd.DataFrame({'review': documents, 'sentiment': labels})
df['label'] = (df['sentiment'] == 'pos').astype(int)  # 1=positive, 0=negative

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['sentiment'].value_counts())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
colors = ['#e74c3c', '#2ecc71']
df['sentiment'].value_counts().plot(
    kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85
)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Review length distribution
df['review_length'] = df['review'].apply(lambda x: len(x.split()))
for sentiment, color in zip(['pos', 'neg'], colors):
    subset = df[df['sentiment'] == sentiment]['review_length']
    axes[1].hist(subset, bins=40, alpha=0.6, color=color, label=sentiment, edgecolor='black')
axes[1].set_title('Review Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nAverage review length: {df['review_length'].mean():.0f} words")

In [ ]:
# Peek at a sample review
print("--- POSITIVE REVIEW SAMPLE ---")
print(df[df['sentiment']=='pos']['review'].iloc[0][:500], "...\n")

print("--- NEGATIVE REVIEW SAMPLE ---")
print(df[df['sentiment']=='neg']['review'].iloc[0][:500], "...")

## 3. Text Preprocessing

We apply the following pipeline:
- Lowercase
- Remove punctuation & numbers
- Tokenize
- Remove stopwords
- Lemmatize

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    """Full text preprocessing pipeline."""
    # Lowercase
    text = text.lower()
    # Remove HTML tags if any
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

print("Preprocessing reviews...")
df['cleaned'] = df['review'].apply(preprocess)
print("✅ Done!")

# Show before/after
print("\n--- Before preprocessing ---")
print(df['review'].iloc[0][:200])
print("\n--- After preprocessing ---")
print(df['cleaned'].iloc[0][:200])

## 4. Feature Extraction

We compare two approaches:
- **Bag of Words (BoW):** counts word occurrences
- **TF-IDF:** weighs words by how unique they are across documents

In [ ]:
X = df['cleaned']
y = df['label']

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Bag of Words
bow_vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow  = bow_vectorizer.transform(X_test)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print(f"Train size : {X_train_bow.shape[0]} samples")
print(f"Test size  : {X_test_bow.shape[0]} samples")
print(f"Vocabulary : {X_train_bow.shape[1]} features")

## 5. Model Training & Comparison

We train 4 classifiers on both feature sets and compare performance.

In [ ]:
models = {
    'Naive Bayes'        : MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'Linear SVM'         : LinearSVC(max_iter=2000, C=1.0),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    for feat_name, X_tr, X_te in [
        ('BoW',   X_train_bow,   X_test_bow),
        ('TF-IDF', X_train_tfidf, X_test_tfidf)
    ]:
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_tr, y_train)
        preds = model_clone.predict(X_te)
        acc = accuracy_score(y_test, preds)
        results.append({
            'Model': name,
            'Features': feat_name,
            'Accuracy': round(acc * 100, 2)
        })
        print(f"{name:22s} | {feat_name:6s} | Accuracy: {acc*100:.2f}%")

results_df = pd.DataFrame(results)

In [ ]:
# Visualize model comparison
fig, ax = plt.subplots(figsize=(11, 5))

pivot = results_df.pivot(index='Model', columns='Features', values='Accuracy')
pivot.plot(kind='bar', ax=ax, color=['#3498db', '#e67e22'], edgecolor='black', alpha=0.85, width=0.65)

ax.set_title('Model Accuracy: BoW vs TF-IDF Features', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)
ax.set_ylim(70, 100)
ax.legend(title='Feature Type')
ax.grid(axis='y', alpha=0.4)

# Annotate bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best Model — Detailed Evaluation

In [ ]:
# Train best model (Logistic Regression + TF-IDF is typically best)
best_model = LogisticRegression(max_iter=1000, C=1.0)
best_model.fit(X_train_tfidf, y_train)
y_pred = best_model.predict(X_test_tfidf)

print("=" * 45)
print("  BEST MODEL: Logistic Regression + TF-IDF")
print("=" * 45)
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# Top predictive words
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
coefs = best_model.coef_[0]
top_n = 15

top_pos_idx = np.argsort(coefs)[-top_n:]
top_neg_idx = np.argsort(coefs)[:top_n]

words = np.concatenate([feature_names[top_neg_idx], feature_names[top_pos_idx]])
weights = np.concatenate([coefs[top_neg_idx], coefs[top_pos_idx]])
colors_bar = ['#e74c3c' if w < 0 else '#2ecc71' for w in weights]

axes[1].barh(range(len(words)), weights, color=colors_bar, edgecolor='black', alpha=0.8)
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words, fontsize=9)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Most Predictive Words\n(Red = Negative | Green = Positive)',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coefficient Weight')

plt.tight_layout()
plt.savefig('outputs/best_model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Error Analysis

Understanding *where* the model fails is as important as the accuracy score.

In [ ]:
# Get misclassified examples
X_test_text = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

errors_df = pd.DataFrame({
    'review': X_test_text,
    'true_label': y_test_reset.map({1: 'positive', 0: 'negative'}),
    'predicted' : pd.Series(y_pred).map({1: 'positive', 0: 'negative'})
})

errors_df = errors_df[errors_df['true_label'] != errors_df['predicted']]
print(f"Total misclassifications: {len(errors_df)} / {len(y_test)} ({len(errors_df)/len(y_test)*100:.1f}%)\n")

print("--- FALSE POSITIVE (predicted positive, actually negative) ---")
fp = errors_df[errors_df['predicted'] == 'positive'].iloc[0]
print(fp['review'][:400], "...\n")

print("--- FALSE NEGATIVE (predicted negative, actually positive) ---")
fn = errors_df[errors_df['predicted'] == 'negative'].iloc[0]
print(fn['review'][:400], "...")

## 8. Live Prediction Demo

In [ ]:
def predict_sentiment(text):
    """Predict sentiment of a custom review."""
    cleaned = preprocess(text)
    vec = tfidf_vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    prob = best_model.predict_proba(vec)[0] if hasattr(best_model, 'predict_proba') else None
    label = '😊 POSITIVE' if pred == 1 else '😞 NEGATIVE'
    if prob is not None:
        confidence = max(prob) * 100
        print(f"Sentiment  : {label}")
        print(f"Confidence : {confidence:.1f}%")
    else:
        print(f"Sentiment  : {label}")

# Test it!
print("=" * 40)
predict_sentiment("""
    This film was absolutely brilliant. The acting was superb and the storyline
    kept me gripped from start to finish. A masterpiece!
""")

print("\n" + "=" * 40)
predict_sentiment("""
    Terrible movie. Boring plot, awful acting, complete waste of time.
    I fell asleep halfway through.
""")

## 9. Summary & Key Findings

| Model | BoW Accuracy | TF-IDF Accuracy |
|---|---|---|
| Naive Bayes | — | — |
| Logistic Regression | — | — |
| Linear SVM | — | — |
| Random Forest | — | — |

*(Fill in with actual values after running)*

### Observations
- **TF-IDF consistently outperforms BoW** because it downweights common, uninformative words
- **Logistic Regression and SVM** are the strongest classical models for this task
- **Naive Bayes** is fast and surprisingly competitive despite its independence assumption
- **Error analysis** shows the model struggles with sarcasm and ambiguous reviews

### Possible Improvements
- Use pre-trained word embeddings (Word2Vec, GloVe)
- Fine-tune a transformer model (BERT, DistilBERT)
- Incorporate negation handling in preprocessing
- Hyperparameter tuning via GridSearchCV

---
*This project is part of my AI/ML portfolio. See more at [github.com/mdarifhasanbadsha](https://github.com/mdarifhasanbadsha)*